# Week 1 — Theory
## Scientific data types, magnitudes, and distributions

This notebook builds the conceptual foundation for the rest of the week. We will:

1. Lay out the **measurement-type taxonomy** that determines which encodings are even
   legal for a variable.
2. Look at the **perceptual ranking** of visual channels (Cleveland & McGill) and use it
   to make defensible chart choices.
3. Work through **scale selection** — linear, log, symlog — for quantities that span
   many orders of magnitude, which is the rule rather than the exception in AI work.
4. Compare the families of **distribution plots** (histogram, KDE, ECDF, box, violin,
   strip/swarm, ridgeline) and discuss when each is the most honest answer.
5. End with a short section on **presentation ethics**: the failure modes that show up
   over and over in AI papers, dashboards, and marketing material.

> **Setup.** Run the cell below once. The rest of the notebook assumes the style is loaded
> and the seed is fixed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(7)

# A small palette mirroring the style file
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]


## 1. Measurement types and what they let you do

| Type     | Examples                                  | Allowed operations           | Sensible encodings |
|----------|-------------------------------------------|------------------------------|--------------------|
| Nominal  | model family, benchmark name              | `=`, `≠`                     | colour hue, shape, position-on-categorical-axis |
| Ordinal  | severity level, satisfaction (1–5)        | `=`, `<`, `>`                | position on ordered axis, shape size with care |
| Interval | calendar year, temperature in °C          | `+`, `−`                     | position on linear axis |
| Ratio    | accuracy, latency, parameter count        | `+`, `−`, `×`, `÷`, `log()`  | position on linear or log axis |

This sounds pedantic until you remember that **a colour gradient is only meaningful on an
ordered variable**, and **a ratio of "accuracy" makes sense while a ratio of "year" does
not**. Most "the chart is misleading" arguments at conference posters reduce to a
violation of this table.

Two concrete consequences for AI work:

- **Parameter count** is a ratio variable that spans roughly four orders of magnitude.
  Encoding it on a linear x-axis squashes the small models into a single column. Use a
  log axis or bin the variable into ordinal buckets before plotting.
- **Benchmark name** is nominal. There is no "natural" order on the x-axis — pick one
  (e.g. by mean difficulty) and tell the reader what you did, otherwise the visual order
  itself is misleading.


## 2. The perceptual ranking of visual channels

Cleveland & McGill (1984) ran the experiments that everyone still cites. The takeaway,
ranked from most to least accurately perceived:

1. Position along a common scale (e.g. scatter, dot plot)
2. Position along non-aligned scales (small multiples)
3. Length (bar height)
4. Angle / slope
5. Area
6. Colour saturation / value
7. Colour hue
8. Volume

Two practical rules drop out:

- **Encode your most important variable in position.** Everything else is decoration.
- **Pie charts and donut charts encode in angle and area**, both of which sit below half
  of the list. A grouped bar chart is almost always strictly better.

We illustrate this with a small experiment: read the values off the four charts below.
The differences are identical in every chart — only the encoding changes.

In [ ]:
values = [0.62, 0.71, 0.66, 0.73, 0.68]
labels = ["A", "B", "C", "D", "E"]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))

# (1) Position on common scale — dot plot
axes[0].scatter(values, range(len(values)), s=80, color=PALETTE[0])
axes[0].set_yticks(range(len(values)), labels)
axes[0].set_xlim(0.55, 0.80)
axes[0].set_xlabel("accuracy")
axes[0].set_title("(1) Position on common scale")

# (2) Length — bar
axes[1].bar(labels, values, color=PALETTE[0])
axes[1].set_ylim(0, 1)
axes[1].set_title("(2) Length (bar from 0)")

# (3) Angle — pie
axes[2].pie(values, labels=labels, colors=PALETTE)
axes[2].set_title("(3) Angle / area (pie)")

# (4) Colour hue — heat strip
axes[3].imshow(np.array(values).reshape(1, -1), cmap="viridis", aspect="auto")
axes[3].set_xticks(range(len(values)), labels)
axes[3].set_yticks([])
axes[3].set_title("(4) Colour hue")

plt.tight_layout()
plt.show()


Looking at chart (1) you can read each value to about one or two percentage points.
Chart (3) — the pie chart — you cannot. The information content is identical; the
perceptual cost is not.

A common counterargument is "pie charts are fine for *proportions of a whole*". They are
defensible there, but a stacked bar still wins for any comparison across categories. We
will return to this in the **proportions** section of the lab.

## 3. Magnitudes and scale selection

Three quantities you will see constantly in AI research and that all need careful scaling:

- **Training loss** — falls by orders of magnitude in the first few hundred steps and
  then crawls. Log-y is mandatory for inspecting late-training dynamics.
- **Parameter count** — spans roughly $10^8$ to $10^{12}$ across modern models. Log-x is
  mandatory.
- **Cost per query / latency** — typically log-normal. Geometric means and log axes are
  the honest default; arithmetic means hide tail behaviour.

In [ ]:
# Train loss simulation — power-law decay with noise floor
steps = np.arange(1, 5000)
loss = 8.0 * steps ** (-0.35) + 0.18 + RNG.normal(0, 0.02, size=steps.size)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(steps, loss, color=PALETTE[0])
axes[0].set(xlabel="step", ylabel="loss", title="Linear y — late training looks 'flat'")

axes[1].plot(steps, loss, color=PALETTE[0])
axes[1].set_yscale("log")
axes[1].set(xlabel="step", ylabel="loss (log)", title="Log y — the noise floor is visible")

plt.tight_layout()
plt.show()


The two panels show **the same data**. On a linear scale, the run looks like it
"converged" around step 1000 — the eye flattens anything within a small absolute
distance of the asymptote. On a log scale you can see that the loss continues to drop
modestly, and you can read off the level of the noise floor.

### When to use symlog

`symlog` is log away from zero and linear in a band around it. Use it when:

- the variable can be **negative** (residuals, attention deltas, log-prob differences),
  or
- the variable can be **exactly zero** in a meaningful fraction of points (e.g.
  attribution scores).

`linthresh` controls the width of the linear band. Pick it slightly larger than the
typical noise level of the quantity.

In [ ]:
# Symlog example — residuals that span both signs and many magnitudes
x = np.linspace(0, 10, 400)
residuals = np.sin(x) * np.exp(x / 3) * RNG.choice([-1, 1], size=x.size)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(x, residuals, color=PALETTE[1])
axes[0].set(xlabel="x", ylabel="residual", title="Linear y — small residuals invisible")
axes[0].axhline(0, color="k", lw=0.6)

axes[1].plot(x, residuals, color=PALETTE[1])
axes[1].set_yscale("symlog", linthresh=0.5)
axes[1].set(xlabel="x", ylabel="residual (symlog)", title="Symlog — sign preserved, range visible")
axes[1].axhline(0, color="k", lw=0.6)
plt.tight_layout()
plt.show()


## 4. The distribution-plot family

A working AI researcher needs to be fluent in at least six distribution plots. Each one
makes a different trade-off between **fidelity** (how much of the distribution is shown)
and **legibility** (how many distributions you can compare in a single figure).

In [ ]:
# Synthetic accuracies from three benchmarks
np.random.seed(0)
data = pd.DataFrame({
    "benchmark": np.repeat(["mmlu", "gsm8k", "math"], 80),
    "accuracy": np.concatenate([
        RNG.normal(0.62, 0.05, 80),
        RNG.normal(0.42, 0.09, 80),
        RNG.beta(2, 6, 80) * 0.6,   # skewed
    ]),
})

fig, axes = plt.subplots(2, 3, figsize=(13, 7))

sns.histplot(data=data, x="accuracy", hue="benchmark", element="step",
             stat="density", common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title("Histogram")

sns.kdeplot(data=data, x="accuracy", hue="benchmark", common_norm=False, ax=axes[0, 1])
axes[0, 1].set_title("KDE")

sns.ecdfplot(data=data, x="accuracy", hue="benchmark", ax=axes[0, 2])
axes[0, 2].set_title("ECDF — no bandwidth choice")

sns.boxplot(data=data, x="benchmark", y="accuracy", ax=axes[1, 0])
axes[1, 0].set_title("Box — five-number summary")

sns.violinplot(data=data, x="benchmark", y="accuracy", inner="quartile", ax=axes[1, 1])
axes[1, 1].set_title("Violin — KDE on its side")

sns.stripplot(data=data, x="benchmark", y="accuracy", alpha=0.6, ax=axes[1, 2])
axes[1, 2].set_title("Strip — every datum visible")

for ax in axes.flat:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


**When to pick which?**

- **Histogram** is the right default for a *single* distribution. Bin choice matters;
  Sturges' rule is a defensible starting point.
- **KDE** smooths and is easier on the eye for overlays, but it imposes a kernel and a
  bandwidth. Both choices can change the apparent shape — be suspicious of strong
  conclusions drawn from a single KDE.
- **ECDF** has no bandwidth, no binning, no interpretation overhead. If you only learn
  one plot from this week, learn this one.
- **Box plot** is great for many groups at once. Its weakness: it hides bimodality.
- **Violin** fixes the bimodality problem but reintroduces the bandwidth problem.
- **Strip / swarm** are the best choice when *n* is small (say, < 200 per group). They
  show every data point, including outliers and discreteness artefacts.

### The ECDF deserves its own moment

The empirical CDF answers the most natural question you can ask of a distribution:
"what fraction of the data is below value $x$?" It is invariant to monotone
transformations of $x$, easy to overlay across groups, and impossible to mislead with
bandwidth choice.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.ecdfplot(data=data, x="accuracy", hue="benchmark", ax=ax, lw=2)
ax.axhline(0.5, color="gray", lw=0.7, ls="--")
ax.text(0.02, 0.52, "median line", color="gray", fontsize=9)
ax.set_title("ECDF — read off any quantile by eye")
plt.show()


## 5. Presentation ethics: failure modes in AI charts

A non-exhaustive list of patterns that show up in ML papers and dashboards regularly.
They all *look* like reasonable visualization choices on the surface.

### 5.1 The truncated y-axis

The classic. The chart starts the y-axis at, say, 60 % accuracy instead of 0, and a
1-percentage-point improvement looks like a doubling.

In [ ]:
before, after = 0.812, 0.823

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(["baseline", "ours"], [before, after], color=[PALETTE[5], PALETTE[1]])
axes[0].set_ylim(0.80, 0.83)
axes[0].set_title("Truncated y — same gap looks enormous")
axes[0].set_ylabel("accuracy")

axes[1].bar(["baseline", "ours"], [before, after], color=[PALETTE[5], PALETTE[1]])
axes[1].set_ylim(0, 1)
axes[1].set_title("Honest y — same gap, true size")
axes[1].set_ylabel("accuracy")

for ax in axes:
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=2)
plt.tight_layout()
plt.show()


Both panels are technically truthful. The right one tells the reader what 1.1
percentage points of accuracy actually looks like. The left one tells the reader the
authors really wanted them to be impressed.

**Rule of thumb.** For bar charts of ratio variables, start at zero unless you have a
specific reason and you say so in the caption.

### 5.2 Averaging seeds without showing variance

A model trained with three seeds and a single number per configuration is barely an
experiment. Show the spread.

In [ ]:
# Two methods, three seeds, same mean — different stories
mean = 0.72
method_a = [0.715, 0.720, 0.725]
method_b = [0.65,  0.72,  0.79 ]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(["A", "B"], [np.mean(method_a), np.mean(method_b)],
            color=PALETTE[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("Mean only — methods look identical")

axes[1].errorbar(["A", "B"], [np.mean(method_a), np.mean(method_b)],
                 yerr=[np.std(method_a, ddof=1), np.std(method_b, ddof=1)],
                 fmt="o", color=PALETTE[0], capsize=6)
# overlay raw points
for i, vals in enumerate([method_a, method_b]):
    axes[1].scatter([i] * len(vals), vals, color=PALETTE[1], alpha=0.7, zorder=3)
axes[1].set_ylim(0, 1)
axes[1].set_title("Mean ± SD with raw seeds — method B is unstable")
plt.tight_layout()
plt.show()


### 5.3 The "single number per benchmark" leaderboard

A leaderboard row that reports a single accuracy per benchmark obscures everything
interesting: variance across seeds, behaviour by question difficulty, robustness to
paraphrase, calibration. It is the visualization equivalent of `print(metric.mean())`.

For research-grade reporting:

- show **per-seed** numbers (strip plot is fine),
- separate **in-distribution** from **out-of-distribution** test sets,
- report at least one **uncertainty estimate** (bootstrap CI is cheap and assumption-light).

### 5.4 Misleading colormaps

`jet` and `rainbow` are still in widespread use. They are not perceptually uniform —
they create false visual boundaries where the data has none, and they are unreadable
when printed in black and white or by colour-blind readers.

Use a perceptually uniform colormap (`viridis`, `magma`, `cividis`) for sequential data,
and a diverging colormap with a meaningful midpoint (`RdBu_r`, `coolwarm`) for signed
data.

In [ ]:
# The same noisy gradient under four colormaps
img = np.linspace(0, 1, 300).reshape(1, -1) + RNG.normal(0, 0.02, (1, 300))

fig, axes = plt.subplots(4, 1, figsize=(8, 3.5))
for ax, cmap in zip(axes, ["jet", "rainbow", "viridis", "cividis"]):
    ax.imshow(img, aspect="auto", cmap=cmap)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(cmap, rotation=0, labelpad=30, va="center")
plt.tight_layout()
plt.show()


`viridis` and `cividis` show the gradient as a smooth ramp. `jet` and `rainbow`
manufacture visual bands at green/yellow and yellow/red transitions — bands that are not
in the data.

## Summary

- Pin down the **measurement type** of each variable before you reach for a plot.
- Encode the most important variable using **position on a common scale**.
- Use a **log axis** when the variable spans many orders of magnitude; use **symlog**
  when it spans many orders of magnitude *and* changes sign.
- Prefer **ECDFs** over histograms for comparing distributions across groups.
- Prefer **violins or strips** over box plots when the distribution might be multimodal
  or when group sizes are small.
- Be paranoid about **truncated axes, averaged-away variance, and `jet`**.

In the lab notebook we apply all of this to an actual LLM-evaluation results table.
